# Lesson 8.3 — One Tomato Through Ten Modules

Trace a single fruit through the whole stack: a position (M1) in a frame (M2), seen by perception (M3), with forward kinematics (M4), inverse kinematics (M5), the Jacobian (M6), a planned reach (M7), control (M8), the integrated pick (M9), and the twin that mirrors it (M10).

In [1]:
import numpy as np
from modules.module09 import integration as I
from modules.module10.twin import DigitalTwin, snapshot, monitor
mk = lambda: I.GreenhouseWorld.demo_row(n=5, seed=1)
checks = []
w = mk(); fruit = w.fruit[0]
# M1 — represent the tomato's position as a vector
p = np.array([float(fruit.x), float(fruit.y)]); print('M1 vector p =', np.round(p, 3))
checks.append(p.shape == (2,))


M1 vector p = [0.22  0.003]


In [2]:
# M3 — perception sees the fruit (detections -> understanding)
det = I.model_perception(w); und = I.understand(det, w)
print('M3 detections =', len(det))
checks.append(len(det) >= 1)


M3 detections = 5


In [3]:
# M5 — inverse kinematics: joint angles that put the tip on the tomato
q = I.ik_2link(p[0], p[1]); print('M5 joint angles q =', np.round(q, 3))
# M4 — forward kinematics confirms the tip lands on the tomato
tip = np.asarray(I.fk_xy(I.P2, I.T2, q)); print('M4 FK tip =', np.round(tip, 3))
checks.append(float(np.linalg.norm(tip - p)) < 1e-9)   # IK<->FK round-trip


M5 joint angles q = [-0.821  2.568]
M4 FK tip = [0.22  0.003]


In [4]:
# M6 — the Jacobian relates joint rates to tip motion (twist [v; omega])
J = np.asarray(I.geometric_jacobian(I.P2, I.T2, q)); print('M6 Jacobian shape =', J.shape)
checks.append(J.shape[1] == 2)            # two joints


M6 Jacobian shape = (6, 2)


In [5]:
# M7-M9 — plan, control, and the integrated harvest actually pick the row
rep = I.harvest_row(mk(), max_attempts=3)
print('M9 harvested =', sorted(rep['harvested']), '| complete =', rep['complete'])
checks.append(fruit.fid in rep['harvested'])   # our tomato is among the picked
checks.append(rep['complete'] is True)


M9 harvested = ['F0', 'F1', 'F2', 'F3'] | complete = True


In [6]:
# M10 — the twin mirrors the run and monitors it (in sync after a fresh sync)
twin = DigitalTwin(mk()); twin.sync(snapshot(mk()))
m = monitor(twin, snapshot(mk()))
print('M10 monitor alert =', m['alert'], '(in sync)')
checks.append(m['alert'] is False)


M10 monitor alert = False (in sync)


In [7]:
# Ten modules, one tomato, one pipeline.
assert all(checks), checks
print('All checks passed.')


All checks passed.
